# Limma解析

【エクスポートデータ】 ・DEP_list.csv：有意差基準を満たした変動タンパク質のリスト
。 ・result_labels.csv：全タンパク質の統計解析結果テーブル
。 ・target_labels.csv：ROC分析等で注目した特定マーカーのラベル情報
。

# Import Library

In [ ]:
install.packages("tidyverse")

In [ ]:
library("tidyverse")
options(repr.plot.width=5, repr.plot.height=5)#Default=7x7

if (!require("BiocManager", quietly = TRUE))
    install.packages("BiocManager")


for (package in c("DEqMS","WGCNA")) {
    BiocManager::install(package)
    eval(bquote(library(.(package))))
    print(str_c(package, ": ", as.character(packageVersion(package))))
}
#CRAN
for (package in c("RColorBrewer", "ggrepel")) {
    install.packages(package)
    eval(bquote(library(.(package))))
    print(str_c(package, ": ", as.character(packageVersion(package))))
}

# Loading Data

In [ ]:
fileDir <- "./Data/"
fileName <- "pro_data_common_dis.csv"

Path <- str_c(fileDir, fileName)
temp <- read.csv(Path, row.names = 1)
head(temp)
pro_data_dis <- temp
dim(temp)

fileDir <- "./Data/"
fileName <- "pro_sample_dis.csv"

Path <- str_c(fileDir, fileName)
temp <- read.csv(Path)
colnames(temp)[1] <- "SampleID"
head(temp)
pro_sample_dis <- temp


fileDir <- "./Data/"
fileName <- "pro_data_common_val.csv"

Path <- str_c(fileDir, fileName)
temp <- read.csv(Path, row.names = 1)
head(temp)
pro_data_val <- temp
dim(temp)

fileDir <- "./Data/"
fileName <- "pro_sample_val.csv"

Path <- str_c(fileDir, fileName)
temp <- read.csv(Path)
colnames(temp)[1] <- "SampleID"
head(temp)
pro_sample_val <- temp


In [ ]:
pro_data_val_ori <- pro_data_val

In [ ]:
map <- read.csv("./Data/map_OID_Uni_Sym.csv", row.names = 1)

# UniProtをキーにしてSymbolを取得
uni_to_symbol <- setNames(map$Symbol, map$UniProt)

temp <- pro_data_val
# pro_data_valの列名をSymbolに変換
colnames(temp) <- ifelse(
  colnames(temp) %in% names(uni_to_symbol),
  uni_to_symbol[colnames(temp)],
  colnames(temp)
)
head(temp)
pro_data_val <- temp
head(pro_data_val)

## 重複したIDを確認

In [ ]:
duplicated_names <- colnames(pro_data_val)[duplicated(colnames(pro_data_val))]

which(colnames(pro_data_val) == "PALM2AKAP2")
pro_data_val[, which(colnames(pro_data_val) == "PALM2AKAP2")]

In [ ]:
# Symbolが重複しているものを抽出
map_dup <- map[
  duplicated(map$Symbol) | duplicated(map$Symbol, fromLast = TRUE),
  c("OlinkID", "UniProt", "Symbol")
]

# Symbolごとに並べる
map_dup <- map_dup[order(map_dup$Symbol), ]

map_dup

In [ ]:
# UniProt → Symbol
uni_to_symbol <- setNames(map$Symbol, map$UniProt)

# pro_data_valのUniProt列名を取得
uniprot_names <- colnames(pro_data_val_ori)

# Symbolに変換
symbol_names <- ifelse(
  uniprot_names %in% names(uni_to_symbol),
  uni_to_symbol[uniprot_names],
  uniprot_names
)

# 重複しているSymbolを確認
dup_symbols <- unique(symbol_names[duplicated(symbol_names)])

# 重複しているSymbolだけ、Symbol(UniProt)にする
for (i in seq_along(symbol_names)) {
  if (symbol_names[i] %in% dup_symbols) {
    symbol_names[i] <- paste0(
      symbol_names[i],
      "(",
      uniprot_names[i],
      ")"
    )
  }
}

temp <- pro_data_val
# 列名を変更
colnames(temp) <- symbol_names
head(temp)
colnames(temp)[grepl("^PALM2AKAP2", colnames(pro_data_val))]
pro_data_val <- temp

In [ ]:
write.csv(pro_data_val, file = "Data/pro_data_common_val_Assay.csv", row.names = TRUE)

# Design Matrix

In [ ]:
sample_group <- c()
sample_gender <- c()
sample_age <- c()
sample_bmi <- c()

for (sample in rownames(pro_data_val)) {
    # group <- pro_sample_val[rownames(pro_data_val) %in% pro_sample_val$SampleID,]$CaCo_Status
    matched_row <- pro_sample_val[pro_sample_val$SampleID == sample, ]
    
    group <- if(nrow(matched_row) > 0) matched_row$CaCo_Status else NA
    sample_group <- c(sample_group, group)
    
    group <- if(nrow(matched_row) > 0) matched_row$sex else NA
    sample_gender <- c(sample_gender, group)
    
    group <- if(nrow(matched_row) > 0) matched_row$fvage else NA
    sample_age <- c(sample_age, group)
    
    group <- if(nrow(matched_row) > 0) matched_row$bmi else NA
    sample_bmi <- c(sample_bmi, group)
    
    # print(sample_group)
    # table(rownames(pro_data_val) %in% pro_sample_val$SampleID)
}
group_order <- c(0, 1)
sample_group <- factor(sample_group, levels = group_order, labels = c("Non.affected", "Affected"))
group_order <- c(1,2)
sample_gender <- factor(sample_gender, levels = group_order, labels = c("Male", "Female"))

design <- model.matrix(~0 + sample_group + sample_gender + sample_age + sample_bmi)
colnames(design) <- c("Non.affected", "Affected", "Female", "Age", "BMI")
design

## Check

In [ ]:
#check
tempD <- apply(design, 2, sum)
tempD <- tibble(Group = names(tempD), DesignMatrix = tempD)
tempD2 <- pro_sample_val
# print(tempD)
# str(tempD2$CaCo_Status)

tempD2 %>%
    dplyr::mutate(Group = factor(CaCo_Status, levels = c(0, 1), labels = c("Non.affected", "Affected"))) %>%
    # dplyr::mutate(Group = as.character(CaCo_Status)) %>%
    dplyr::group_by(Group) %>%
    dplyr::summarize(nSamples = n()) %>%
    dplyr::left_join(tempD, by = "Group")

## Result of Limma

In [ ]:
dim(t(pro_data_val))
dim(design)

In [ ]:
#Limmaでやってみる
library(limma)
pro_data_val_t <- t(pro_data_val)


fit <- lmFit(pro_data_val_t, design)
contrast <- makeContrasts(
    Affected_vs_Non_Affected = Affected - Non.affected,
    levels = design
)
fit2 <- contrasts.fit(fit, contrast)
fit2 <- eBayes(fit2)
result_table <- topTable(fit2,coef = "Affected_vs_Non_Affected", number = Inf, adjust.method = "BH")

# rownames(result_table) <- result_table[[1]]

# result_table <- result_table[, -1]
head(result_table)
sig_raw <- sum(result_table$P.Value < 0.05)
sig_adj <- sum(result_table$adj.P.Val < 0.05)

cat("Number of p value < 0.05:", sig_raw, "\n")
cat("Number of Adjusted p value < 0.05:", sig_adj, "\n")
sum(result_table$adj.P.Val < 0.05 & abs(result_table$logFC) > 0.5)

# Export result data

In [ ]:
temp <- result_table %>%
  mutate(
    DEP = case_when(
      adj.P.Val < 0.05 & logFC > 0.5  ~ "Up",
      adj.P.Val < 0.05 & logFC < -0.5 ~ "Down",
      TRUE ~ "NS"
    )
  )
head(temp)
DEPlist <- temp %>%
    filter(DEP != "NS") %>%
    arrange(adj.P.Val)


dim(DEPlist)
DEPlist

#Export DEP list

write.csv(DEPlist, file = "Data/DEP_list.csv", row.names = TRUE)


In [ ]:
library(ggrepel)
plot_data <- result_table %>%
  mutate(
    threshold = case_when(
      adj.P.Val < 0.05 & logFC > 0.5  ~ "Up",
      adj.P.Val < 0.05 & logFC < -0.5 ~ "Down",
      TRUE ~ "NS"
    )
  )

label_data <- bind_rows(
    data = plot_data %>% filter(adj.P.Val < 0.05) %>% arrange(adj.P.Val) %>% head(10),
    data = plot_data %>% filter(adj.P.Val < 0.05) %>% arrange(desc(abs(logFC))) %>% head(10)
) %>% distinct()

p <- ggplot(plot_data, aes(x = logFC, y = -log10(P.Value), color = threshold)) +
  geom_point(alpha = 1, size = 1.5) +
  scale_color_manual(values = c("Up" = "#E41A1C", "Down" = "#377EB8", "NS" = "grey")) +
  
  geom_text_repel(
    data = label_data,
    aes(label = rownames(label_data)),
    size = 5,
    box.padding = 0.5,
    max.overlaps = Inf
  ) +
  
  geom_vline(xintercept = c(-0.5, 0.5), linetype = "dashed", color = "black", alpha = 0.5) +
  geom_hline(yintercept = -log10(0.05), linetype = "dashed", color = "black", alpha = 0.5) +
  
  theme_minimal() +
  labs(
    title = "Volcano Plot: Case vs Control",
    subtitle = "Adjusted for Sex, Age, and BMI",
    x = "log2 Fold Change",
    y = "-log10 P-value"
  ) +
  theme(legend.position = "none")

print(p)

In [ ]:
ggsave(
  filename = "./ExportData/Volcano_Plot_status.png",
  plot = p,
  width = 5,
  height = 5,
  units = "in",
  dpi = 600,
  bg = "white"
)

In [ ]:
label_data <- bind_rows(
    data = result_table %>% filter(adj.P.Val < 0.05) %>% arrange(adj.P.Val) %>% head(7),
    data = result_table %>% filter(adj.P.Val < 0.05) %>% arrange(desc(abs(logFC))) %>% head(7)
) %>% distinct()

target_proteins <- rownames(label_data)

plot_data_long <- pro_data_val[, target_proteins] %>%
  as.data.frame() %>%
  mutate(Group = factor(pro_sample_val$CaCo_Status, 
                        levels = c(0, 1), 
                        labels = c("Non-affected", "Affected"))) %>%
  pivot_longer(cols = -Group, names_to = "Protein", values_to = "Expression")

options(repr.plot.width = 8, repr.plot.height = 6)
ggplot(plot_data_long, aes(x = Group, y = Expression, fill = Group)) +
  geom_boxplot(outlier.shape = NA, alpha = 1, width = 0.6) +
  geom_jitter(width = 0.2, alpha = 0.7, size = 0.7, color = "black") +

  facet_wrap(~ Protein, scales = "free_y", ncol = 4) + 
  scale_fill_manual(values = c("Non-affected" = "royalblue", "Affected" = "tomato")) +
  theme_bw() +
  labs(
    title = "Top Significant Proteins (by P-value and logFC)",
    x = NULL, 
    y = "NPX Value"
  ) +
  theme(
    legend.position = "none",
    strip.text = element_text(face = "bold"),
    axis.text.x = element_text(angle = 45, hjust = 1)
  )

In [ ]:
target_proteins

In [ ]:
library(pROC)
roc_muc16 <- roc(pro_sample_val$CaCo_Status, pro_data_val[, "MUC16"])
roc_cpb1  <- roc(pro_sample_val$CaCo_Status, pro_data_val[, "CPB1"])
roc_CXCL8 <- roc(pro_sample_val$CaCo_Status, pro_data_val[, "CXCL8"])
roc_KRT19 <- roc(pro_sample_val$CaCo_Status, pro_data_val[, "KRT19"])

# AUCの表示
print(roc_muc16$auc)
print(roc_cpb1$auc)
print(roc_CXCL8$auc)
print(roc_KRT19$auc)

# プロット処理
plot(roc_muc16, col = "red", main = "ROC Curve Comparison")
plot(roc_cpb1, col = "blue", add = TRUE)
plot(roc_CXCL8, col = "green", add = TRUE)
plot(roc_KRT19, col = "violet", add = TRUE)
legend("bottomright", legend=c("MUC16", "CPB1","CXCL8","KRT19"), col=c("red", "blue","green","violet"), lwd=2)

## テストデータとtrainingデータに分けて

In [ ]:
target_proteins <- c("MUC16", "KRT19", "CPB1", "CXCL8")

# データの抽出と結合（行名が一致している前提）
temp <- pro_data_val[, target_proteins] %>%
    as.data.frame() %>%
    mutate(Outcome = pro_sample_val$CaCo_Status) # Outcome列としてグループ情報を追加

# 結果の確認
head(temp)

In [ ]:
library(caret)
# 1. データの準備（タンパク質名で直接指定）
target_proteins <- c("MUC16", "KRT19", "CPB1", "CXCL8")
temp <- pro_data_val[, target_proteins] %>%
    as.data.frame() %>%
    mutate(Outcome = pro_sample_val$CaCo_Status)

# 2. データ分割
set.seed(123)
trainIndex <- createDataPartition(temp$Outcome, p = .7, list = FALSE)
train_data <- temp[trainIndex, ]
test_data  <- temp[-trainIndex, ]

# 3. 標準化と学習
preProcValues <- preProcess(train_data[, target_proteins], method = c("center", "scale"))
train_scaled <- predict(preProcValues, train_data)
test_scaled  <- predict(preProcValues, test_data)

# 複数モデル
model_combined <- glm(Outcome ~ ., data = train_scaled, family = binomial)
# MUC16単独モデル
model_muc16 <- glm(Outcome ~ MUC16, data = train_scaled, family = binomial)
test_result_combined <- predict(model_combined, test_scaled, type = "response")
roc_combined <- roc(test_scaled$Outcome, test_result_combined)
cat("Combined AUC:", round(roc_combined$auc, 3), "\n")

# MUC16単独モデルの評価
test_result_muc16 <- predict(model_muc16, test_scaled, type = "response")
roc_muc16 <- roc(test_scaled$Outcome, test_result_muc16)
cat("MUC16 Solo AUC:", round(roc_muc16$auc, 3), "\n")

# プロット
plot(roc_combined, col = "blue", legacy.axes = TRUE,
     main = "ROC Curve Comparison: Combined vs MUC16 Solo")
plot(roc_muc16, col = "red", lwd = 2, add = TRUE)

legend("bottomright", legend = c(
  paste0("Combined (AUC=", round(roc_combined$auc, 3), ")"),
  paste0("MUC16 Solo (AUC=", round(roc_muc16$auc, 3), ")")
), col = c("blue", "red"), lwd = 2)


## 様々な組み合わせで

In [ ]:

# 1. データの準備
target_proteins <- c("MUC16", "KRT19", "CPB1", "CXCL8")

temp <- pro_data_val[, target_proteins] %>%
    as.data.frame() %>%
    mutate(Outcome = pro_sample_val$CaCo_Status)

# 2. データ分割
set.seed(123)
trainIndex <- createDataPartition(temp$Outcome, p = .7, list = FALSE)
train_data <- temp[trainIndex, ]
test_data  <- temp[-trainIndex, ]

# 3. 標準化
preProcValues <- preProcess(train_data[, target_proteins], method = c("center", "scale"))
train_scaled <- predict(preProcValues, train_data)
test_scaled  <- predict(preProcValues, test_data)

# 4. モデルリストの定義（タンパク質名を直接使用）
models_list <- list(
    "MUC16_only"          = "MUC16",
    "MUC16_CXCL8"         = c("MUC16", "CXCL8"),
    "MUC16_KRT19"         = c("MUC16", "KRT19"),
    "MUC16_CPB1"          = c("MUC16", "CPB1"),
    "MUC16_CXCL8_KRT19"   = c("MUC16", "CXCL8", "KRT19"),
    "MUC16_CXCL8_CPB1"    = c("MUC16", "CXCL8", "CPB1"),
    "MUC16_CPB1_KRT19"    = c("MUC16", "CPB1", "KRT19"),
    "All_4_Markers"       = target_proteins
)

# 5. AUC計算関数の定義
get_auc <- function(marker_set) {
  f <- as.formula(paste("Outcome ~", paste(marker_set, collapse = " + ")))
  model <- glm(f, data = train_scaled, family = binomial)
  preds <- predict(model, test_scaled, type = "response")
  return(roc(test_scaled$Outcome, preds, quiet = TRUE)$auc)
}

# 6. 実行と結果表示
results <- sapply(models_list, get_auc)
results_df <- data.frame(Model = names(results), AUC = as.numeric(results))
print(results_df)

# 全変量モデルの係数確認
final_model <- glm(Outcome ~ ., data = train_scaled, family = binomial)
print(summary(final_model)$coefficients)

### 交差検証

In [ ]:
library(caret)
temp <- pro_data_val[, target_proteins] %>%
    as.data.frame() %>%
    mutate(Outcome = pro_sample_val$CaCo_Status)

# 1. 交差検証用データの準備（Outcomeを因子型に変換）
cv_data <- temp %>%
  mutate(Outcome = factor(ifelse(Outcome == 1, "Affected", "Control"), 
                          levels = c("Control", "Affected")))

# 2. 訓練条件の設定 (5分割交差検証)
train_control_rep <- trainControl(
  method = "repeatedcv", 
  number = 5, 
  repeats = 3, # 繰り返し回数を指定するとより安定します
  classProbs = TRUE,
  summaryFunction = twoClassSummary 
)

# 3. モデルの学習（ロジスティック回帰）
set.seed(123) 
model_cv <- train(
  Outcome ~ ., 
  data = cv_data, 
  method = "glm", 
  family = "binomial",
  trControl = train_control_rep, 
  metric = "ROC"
)

print(model_cv)

# Export data

In [ ]:
write.csv(target_proteins, file = "./Data/target_labels.csv", row.names = FALSE)

In [ ]:
write.csv(result_table, file = "./Data/result_labels.csv", row.names = TRUE)